# Lab 2: Vector Database Setup — Primerica Edition
**Tony DeCarlo | Agentic AI Course**

This notebook builds a searchable Primerica knowledge base using FAISS (local) and Pinecone (cloud).
By the end, you'll have an AI-searchable index of your scripts, objections, and product knowledge.

---
### Before you start, you need:
- OpenAI API key → platform.openai.com
- Pinecone API key → pinecone.io (free account)


## Step 1 — Install Dependencies

In [ ]:
!pip install openai tiktoken faiss-cpu pinecone-client langchain langchain-openai langchain-pinecone chromadb python-dotenv --quiet
print("✅ All packages installed")

## Step 2 — Set Your API Keys

⚠️ Replace the values below with your real keys. Do NOT share this notebook publicly after adding keys.

In [ ]:
import os

# Paste your keys here
os.environ['OPENAI_API_KEY'] = 'YOUR_OPENAI_API_KEY'
os.environ['PINECONE_API_KEY'] = 'YOUR_PINECONE_API_KEY'
os.environ['PINECONE_INDEX'] = 'agentic-ai-lab'

print("✅ Keys set")

## Step 3 — Primerica Knowledge Base (Your Corpus)

This is YOUR business content — objections, scripts, product knowledge, recruiting process.
This is what makes this lab actually useful vs generic.

In [ ]:
docs = [
    {"id":"p1", "text":"The FNA is a Financial Needs Analysis. It shows families how much life insurance they actually need based on income, debts, and dependents. It's a fact-finding conversation, not a sales pitch."},
    {"id":"p2", "text":"Primerica recruits through warm market — friends, family, coworkers. The goal is to invite them to a business overview first, not pitch the product. Lead with curiosity, not pressure."},
    {"id":"p3", "text":"Objection: 'I need to think about it.' Response: 'I completely understand. What specifically is holding you back? Is it the cost, the timing, or something else? Let's address it right now so you have everything you need to decide.'"},
    {"id":"p4", "text":"Term life insurance is pure protection at a low cost. Whole life mixes insurance with a savings component, which drives up the price. Primerica's philosophy: Buy Term and Invest the Difference (BTID)."},
    {"id":"p5", "text":"The RVP (Regional Vice President) path requires promoting 3 legs to Regional Leader and maintaining personal production. Consistent weekly FNAs and recruiting are the engine. Most reps hit RVP in 2-4 years with focused effort."},
    {"id":"p6", "text":"Buy Term and Invest the Difference (BTID) is the core Primerica philosophy. Replace expensive whole life with affordable term coverage and put the savings into mutual funds or investments to build long-term wealth."},
    {"id":"p7", "text":"After an FNA, always follow up within 24-48 hours. Families need time to discuss but waiting too long kills momentum. Send a recap text the same day and schedule a follow-up call before you leave."},
    {"id":"p8", "text":"The IBA (Independent Business Application) is the recruiting contract. New reps pay a one-time licensing fee and must pass their provincial life insurance exam. Primerica covers the cost of the first license attempt."},
    {"id":"p9", "text":"Objection: 'I already have insurance through work.' Response: 'Group coverage is great but it usually ends when you leave your job, and it rarely covers your full income replacement need. Can I show you a quick comparison?'"},
    {"id":"p10", "text":"Objection: 'I can't afford it right now.' Response: 'I understand — that's exactly why we need to talk. Most families find they're over-paying for coverage that doesn't even protect them properly. What if we could get you better coverage for the same or less money?'"},
    {"id":"p11", "text":"The warm market list should have at least 100-200 names. Categories: family, friends, coworkers, neighbors, people from church, gym, past jobs, school. Everyone deserves to hear about this."},
    {"id":"p12", "text":"Primerica's POL (Primerica Online) is the rep portal for managing clients, running illustrations, submitting applications, and tracking team production. New reps should get familiar with it in their first week."},
]

queries = [
    "How do I handle the 'I need to think about it' objection?",
    "What is the process to become an RVP?",
    "How do I explain term life vs whole life to a client?",
    "What do I say when someone says they already have insurance through work?",
    "How do I follow up after an FNA?",
]

print(f"✅ Loaded {len(docs)} Primerica knowledge docs and {len(queries)} test queries")

## Step 4 — Generate Embeddings (OpenAI)

This converts your text into numbers (vectors) that can be mathematically compared for similarity.

In [ ]:
from openai import OpenAI

client = OpenAI()
EMB_MODEL = "text-embedding-3-small"  # fast, affordable, 1536 dimensions

def get_embedding(text: str):
    return client.embeddings.create(model=EMB_MODEL, input=text).data[0].embedding

print("Generating embeddings for docs...")
X = [get_embedding(d["text"]) for d in docs]

print("Generating embeddings for queries...")
Q = [get_embedding(q) for q in queries]

dim = len(X[0])
print(f"✅ Done. Embedding dimension: {dim}")

## Step 5 — FAISS: Build Local Index

FAISS is a local vector search library from Meta. Fast, free, runs on your machine. Great for prototyping.

In [ ]:
import faiss
import numpy as np

xb = np.array(X, dtype="float32")
index = faiss.IndexFlatIP(dim)  # Inner Product (cosine similarity when normalized)

# Normalize vectors for cosine similarity
faiss.normalize_L2(xb)
index.add(xb)

print(f"✅ FAISS index built with {index.ntotal} vectors")

## Step 6 — Query FAISS

Now search your Primerica knowledge base with real questions.

In [ ]:
def faiss_search(query_vec, k=3):
    q = np.array([query_vec], dtype="float32")
    faiss.normalize_L2(q)
    D, I = index.search(q, k)
    return D[0], I[0]

print("=" * 60)
print("FAISS SEARCH RESULTS — Primerica Knowledge Base")
print("=" * 60)

for qi, qv in enumerate(Q):
    D, I = faiss_search(qv, k=2)
    print(f"\n🔍 Query: {queries[qi]}")
    for rank, (score, idx) in enumerate(zip(D, I), 1):
        print(f"  {rank}. [{docs[idx]['id']}] score={round(float(score),4)}")
        print(f"     {docs[idx]['text'][:120]}...")

## Step 7 — Save & Reload FAISS Index

In [ ]:
import json

# Save the FAISS index
faiss.write_index(index, "primerica_knowledge.index")

# Save the metadata map (FAISS only stores vectors, not text)
id_map = {i: d for i, d in enumerate(docs)}
with open("primerica_id_map.json", "w") as f:
    json.dump(id_map, f, indent=2)

# Reload and verify
index2 = faiss.read_index("primerica_knowledge.index")
print(f"✅ Saved and reloaded. Vectors in index: {index2.ntotal}")

## Step 8 — Pinecone: Deploy to Cloud

Pinecone is production-grade managed vector storage. This is what you'd use in a real app.

**Before running:** Log into pinecone.io, create a free Serverless index named `agentic-ai-lab` in `us-east-1 (aws)`, dimension `1536`, metric `cosine`.

In [ ]:
import time
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])
index_name = os.environ['PINECONE_INDEX']

# Create index if it doesn't exist
existing = [i["name"] for i in pc.list_indexes()]
if index_name not in existing:
    print(f"Creating index '{index_name}'...")
    pc.create_index(
        name=index_name,
        dimension=dim,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    while True:
        d = pc.describe_index(index_name)
        if d.status["ready"]: break
        time.sleep(2)
    print("Index ready!")
else:
    print(f"Index '{index_name}' already exists.")

pine_index = pc.Index(index_name)
print(f"✅ Connected to Pinecone index: {index_name}")

## Step 9 — Upsert Primerica Docs into Pinecone

In [ ]:
def normalize(v):
    v = np.array(v, dtype="float32")
    n = np.linalg.norm(v)
    return (v / n).tolist() if n > 0 else v.tolist()

vectors = [{
    "id": d["id"],
    "values": normalize(vec),
    "metadata": {"text": d["text"]}
} for d, vec in zip(docs, X)]

pine_index.upsert(vectors=vectors)
print(f"✅ Upserted {len(vectors)} Primerica knowledge docs to Pinecone")

## Step 10 — Query Pinecone

Same queries, now hitting the cloud index.

In [ ]:
def pinecone_search(query_vec, top_k=2):
    res = pine_index.query(
        vector=normalize(query_vec),
        top_k=top_k,
        include_metadata=True
    )
    return res

print("=" * 60)
print("PINECONE SEARCH RESULTS — Primerica Knowledge Base")
print("=" * 60)

for qi, qv in enumerate(Q):
    res = pinecone_search(qv, top_k=2)
    print(f"\n🔍 Query: {queries[qi]}")
    for match in res["matches"]:
        print(f"  [{match['id']}] score={round(match['score'],4)}")
        print(f"  {match['metadata']['text'][:120]}...")

## Step 11 — (Optional) LangChain Retriever

Wire Pinecone into LangChain so you can plug this into an agent or RAG chain later.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

lc_embeddings = OpenAIEmbeddings(model=EMB_MODEL)
store = PineconeVectorStore(index_name=index_name, embedding=lc_embeddings)

results = store.similarity_search("How do I handle objections about price?", k=2)
print("LangChain retriever results:")
for r in results:
    print("-", r.page_content[:150])

## ✅ Lab Complete!

### What you just built:
- A local FAISS index of 12 Primerica knowledge docs
- A cloud Pinecone index with the same docs
- A LangChain retriever ready to plug into an agent

### What this unlocks:
- In Week 9 (RAG), you'll wire this to a language model so it can answer questions *using your docs*
- Eventually: a Primerica AI assistant that knows your scripts, objections, and products — searchable by your whole team

### To expand it:
- Add more docs (all your objection scripts, product one-pagers, training guides)
- Add metadata fields (category: 'objection', 'product', 'recruiting') for filtered search
- Connect to a ChatGPT-style frontend for your downline to use
